---
title: Lab 4. pymetrix as a Python library
course: Example course (pymetrix)
links: [least-squares, instrumental-variables, methods/OLS, methods/IV]
---

# Lab 4. pymetrix as a Python library

The same economy as Labs 1 to 3, seen from code. The point is that pymetrix is an ordinary Python library underneath the forms: datasets and expressions, equations estimated several ways, results that are plain objects, cross-checks against the libraries it wraps, a new estimator added as a plugin, and dialog steps that are ordinary Python.

## 1. Data and expressions

A `Dataset` is a pandas frame with axes and metadata. Expressions use the same language as formulas: lags in brackets, `log`, `dlog`, `diff`. They are evaluated within each entity of a panel.

In [ ]:
import numpy as np

from pymetrix import IV, OLS, Dataset, Equation

data = Dataset.example("sample_macro").with_series(gc="100 * dlog(c)", gy="100 * dlog(y)")
data.eval("c / y").describe().round(3)

In [ ]:
recent = data.select(span=("2015Q1", None))  # a sample is a view of the data, not a copy to keep in step
recent.frame[["c", "y", "gc", "gy"]].tail(4).round(2)

## 2. One equation, three ways

An `Equation` is a specification; methods are settings objects. Output also depends on consumption, so `y` is endogenous: instrumental variables use government spending and last quarter's output as instruments.

In [ ]:
eq = Equation("c ~ y + c[-1]")
fits = {
    "OLS": eq.fit(data),
    "OLS, HAC": eq.fit(data, method=OLS(cov="hac")),
    "IV": eq.fit(data, method=IV(endog=["y"], instruments=["g", "y[-1]"])),
}
{name: round(float(fit.params["y"]), 4) for name, fit in fits.items()}

In [ ]:
fits["IV"].summary()

## 3. Results are plain objects

Coefficients, covariance and statistics are attributes; a summary is a table that converts to pandas; provenance records the sample, method and data behind the numbers.

In [ ]:
fit = fits["OLS, HAC"]
print(dict(zip(fit.params.names, fit.params.se.round(4).tolist())))
print(fit.provenance.sample, fit.provenance.method)
fit.summary().to_frame()

## 4. Checked against the library underneath

pymetrix computes least squares itself, and checks it against statsmodels; `pymetrix.testing.crosscheck` is the same check the test suite runs.

In [ ]:
import statsmodels.api as sm

from pymetrix.testing import crosscheck

frame = data.frame.assign(c_lag=data.eval("c[-1]")).dropna(subset=["c", "y", "c_lag"])
ref = sm.OLS(frame["c"], sm.add_constant(frame[["y", "c_lag"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 3, "use_correction": True})
crosscheck(fit.params.values, ref.params, what="coefficients")
crosscheck(fit.params.se, ref.bse, what="HAC standard errors")
print("pymetrix and statsmodels agree")

## 5. A new estimator as a plugin

A method is a frozen dataclass with an `estimate` method. Registered, it appears in the Estimate dialog's method list, in scripts and in saved projects, and `pymetrix.testing` checks it follows the conventions. This one is ridge regression, written with `pymetrix.toolkit`.

In [ ]:
from dataclasses import dataclass
from typing import Annotated, ClassVar, Literal

from pymetrix import toolkit
from pymetrix.forms import Widget


@dataclass(frozen=True)
class MyRidge:
    """Ridge regression: least squares with a penalty on the size of the coefficients (not the constant)."""

    label: ClassVar[str] = "My ridge"
    kind: ClassVar[str] = "ls"
    topics: ClassVar[tuple[str, ...]] = ("least-squares",)

    alpha: Annotated[float, Widget("number", min=0, max=1e6, help="penalty on the coefficients; 0 = least squares")] = 1.0
    cov: Annotated[Literal["classic", "white"], Widget("choice", help="covariance of the shrunk coefficients")] = "classic"

    def estimate(self, eq, data, *, span=None, where=None, weights=None, missing="drop"):
        d = toolkit.design(eq, data, span=span, where=where, weights=weights, missing=missing)
        X, y = d.X, d.y
        penalised = np.array([name != "const" for name in d.names])
        scale = np.where(penalised, X.std(axis=0), 1.0)
        A = (X / scale).T @ (X / scale) + self.alpha * np.diag(penalised.astype(float))
        beta = np.linalg.solve(A, (X / scale).T @ y) / scale
        u = y - X @ beta
        Ainv = np.linalg.inv(A)
        meat = (X / scale).T @ (X / scale) * (u @ u / d.df_resid)
        V = (Ainv @ meat @ Ainv) / np.outer(scale, scale)
        return toolkit.result(eq, data, d, self, beta, V, u, extra={"alpha": self.alpha})


toolkit.register("methods", MyRidge)
ridge_fits = {alpha: round(float(eq.fit(data, method=MyRidge(alpha=alpha)).params["y"]), 4) for alpha in (0.0, 10.0, 1000.0)}
ridge_fits

In [ ]:
from pymetrix.plugins import registry
from pymetrix.testing import check_entry, check_result

check_entry(registry("methods").get("My ridge"))  # form, help, generated code and settings round trip
check_result(eq.fit(data, method=MyRidge(alpha=10)))  # a core result, with views that render
crosscheck(eq.fit(data, method=MyRidge(alpha=0)).params.values, fits["OLS"].params.values, what="ridge with no penalty is OLS")
print("MyRidge passes the plugin checks")

## 6. Dialog steps are ordinary Python

The forms run commands; each command records the Python it stands for. Running that Python gives the same result, and a saved project rebuilds it exactly.

In [ ]:
from pathlib import Path
import tempfile

from pymetrix.commands import FitEquation, Session, load_project, save_project
from pymetrix.testing import fingerprint

s = Session()
s.objects["data"] = data
s.run(FitEquation("c ~ y + c[-1]", cov="hac", name="cons"))
script = s.script()
print(script)

In [ ]:
namespace = {"data": data}
exec(script, namespace)  # the script, run as plain Python
same_code = np.allclose(namespace["cons"].params.values, s["cons"].params.values)

reopened = load_project(save_project(s, Path(tempfile.mkdtemp()) / "lab4.pymx"))
same_project = fingerprint(reopened["cons"]) == fingerprint(s["cons"])  # every table the result's window shows
print(f"script gives the same result: {same_code}; reopened project identical: {same_project}")

**Question.** What would it take to add your own method this way, and what does `check_entry` stop you getting wrong? (See `docs/adding-features.md` in the repository.)

## 7. A model estimated from data

Equations can also form a model with expectations. The New Keynesian template is solved, its
variables simulated as data, and its parameters estimated back by maximum likelihood with the
Kalman filter: the persistence of the policy rate and the shock sizes come back well; the slope of
the Phillips curve and the response to inflation trade off against each other.

In [ ]:
import pandas as pd

from pymetrix import templates

nk = templates.new_keynesian()
simulated = nk.solve().simulate(400, nk.shock_sd, seed=11)
simulated.index = pd.period_range("1925Q1", periods=400, freq="Q", name="time")
observed = nk.measurement.apply(Dataset.from_frame(simulated))

estimated = templates.new_keynesian(rho=0.6).estimate(observed, estimate=["rho", "sd_ey", "sd_epi", "sd_ei"])
truth = {"rho": 0.8, "sd_ey": 0.5, "sd_epi": 0.3, "sd_ei": 0.2}
table = {n: (round(float(v), 3), round(float(s), 3), truth[n]) for n, v, s in zip(estimated.params.names, estimated.params.values, estimated.params.se)}
assert all(abs(v - truth[n]) < 3 * s for n, (v, s, _) in table.items())  # within three standard errors
table  # estimate, standard error, the value that made the data

**Question.** Add `"kappa"` and `"phi_pi"` to `estimate=` (both start from their true values here). How far
do they move, how large are their standard errors, and what would you do about it: fix one, give
them priors (`priors={"kappa": Gamma(0.1, 0.05)}`), or use more data?